# 04 · Can we train a tiny CLIP ourselves?

In Notebook 03 we rebuilt CLIP's loss on a model someone else had already trained. This time we do the training ourselves: a tiny CLIP, random starting weights, and 800 real image–caption pairs.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 4, Section 4.2.1, *Contrastive Alignment*, and Chapter 2, Section 2.1, *From Pixels to Features: Convolutional Encoders*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens. Training happens exactly once, in Section 6, and none of the TODOs make you retrain unless they say so.

## 1. What do we need before we start?

### 1.1 Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

### 1.2 Grab our tools

Not much here: PyTorch for the model and training, `datasets` for the real COCO captions, CLIP's published tokenizer for chopping text into pieces, a few `torchvision` transforms to resize images, and Matplotlib so we can actually look at what's going on.

In [ ]:
from collections import defaultdict  # handy for grouping several captions under one photo
from textwrap import fill  # wraps long captions so they fit in plot titles

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import CenterCrop, Compose, Resize, ToTensor
from transformers import AutoTokenizer

### 1.3 Pin the data, the tokenizer, and the budget

A **revision** is one exact snapshot (a commit) of a dataset or model on the Hugging Face Hub. Pinning it means you get the same rows and the same token IDs as everyone else running this notebook, even if the repo changes later.

In [ ]:
DATASET_ID = "Erland/coco_captions_small"  # pin the exact snapshot of image-caption pairs
DATASET_REVISION = "a516b5101cec691eb65f455ffdb054afb2fdd5dd"

TOKENIZER_ID = "openai/clip-vit-base-patch32"
TOKENIZER_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"

The revisions fix *which* data and tokenizer we use. The next cell fixes *how big* our experiment is. Here's what each knob means:

| Setting | Value | What it controls |
|---|---|---|
| `TRAIN_SIZE` | 800 | how many image–caption pairs we train on |
| `BATCH_SIZE` | 50 | pairs per mini-batch, so each image competes against 49 wrong captions |
| `EPOCHS` | 40 | how many full passes over the 800 pairs |
| `IMAGE_SIZE` | 64 | images get cropped to 64 × 64 pixels |
| `TOWER_WIDTH` | 128 | how many numbers each tower outputs |
| `EMBED_DIM` | 64 | size of the shared image–text space |
| `LEARNING_RATE`, `WEIGHT_DECAY` | 5e-4, 1e-3 | the optimizer's step size and how hard it shrinks weights |
| `INITIAL_TEMPERATURE` | 0.07 | where the learned temperature starts (the same starting value CLIP used) |

These are kept small on purpose. Tiny 64 × 64 images and a 64-dimensional shared space are what let the whole experiment run on one GPU.

In [ ]:
SEED = 23  # one seed for sampling, weight init, and shuffling
TRAIN_SIZE, BATCH_SIZE, EPOCHS = 800, 50, 40  # want to experiment? see the optional TODO 4 at the end
IMAGE_SIZE, TOWER_WIDTH, EMBED_DIM = 64, 128, 64
LEARNING_RATE, WEIGHT_DECAY = 5e-4, 1e-3
INITIAL_TEMPERATURE = 0.07

We set the seed *before* building the model, so the random starting weights come out the same every run.

Small note: inside this notebook, CUDA device `0` is whichever physical GPU you picked in Section 1.1.

In [ ]:
torch.manual_seed(SEED)  # seed before the model draws its random weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Where do our 800 training pairs come from?

### 2.1 Load real COCO captions

**COCO** (Common Objects in Context) is a big collection of everyday photos, each described by people in a few short captions. We're using a small slice of it.

Here's the catch: COCO gives each photo **several** captions, and each caption gets its own row. So the dataset has more rows than photos.

Why do we care? We're about to do **contrastive learning**, which means learning by contrast: pull each matching image–caption pair together and push the mismatched ones apart. Every other caption in a batch counts as "wrong". If the same photo snuck into a batch twice, one of those "wrong" captions would actually be right. So let's look at the data before we sample anything.

In [ ]:
dataset = load_dataset(  # the pinned revision keeps the pairs identical across runs
    DATASET_ID, split="train", revision=DATASET_REVISION
)
print(dataset)

### 2.2 Group the rows by photo

`cocoid` is the photo's ID. For each ID, we remember the first row (that's where we'll grab the image from) and collect all of its captions. That way no photo can show up twice in our 800 pairs.

In [ ]:
first_row = {}  # photo ID -> index of its first row in the dataset
captions_by_id = defaultdict(list)  # photo ID -> all of its captions

for index, (cocoid, caption) in enumerate(zip(dataset["cocoid"], dataset["caption"])):
    first_row.setdefault(cocoid, index)
    captions_by_id[cocoid].append(caption.strip())

The second number should be smaller than the first, since it counts photos, not caption rows.

In [ ]:
all_ids = list(first_row)  # dicts keep insertion order, so this follows the dataset's order
print("caption rows:", len(dataset))
print("distinct photographs:", len(all_ids))

The gap between those two numbers is the "several captions per photo" thing in action.

### 2.3 Sample photos, then give each one a caption

We sample **photo IDs**, not raw rows, and then attach the first caption of each photo.

Here's the one rule to remember for the whole notebook: position `i` in `train_ids` and position `i` in `captions` are a **positive pair** (a matching image and caption). Everything downstream relies on that order staying lined up.

In [ ]:
sample_generator = torch.Generator().manual_seed(SEED)  # its own generator, so later shuffling can't change which photos we pick
order = torch.randperm(len(all_ids), generator=sample_generator)
train_ids = [all_ids[index] for index in order[:TRAIN_SIZE].tolist()]
captions = [captions_by_id[cocoid][0] for cocoid in train_ids]

### 2.4 Make every image the same size

Our image tower wants a tensor shaped `[batch, channel, height, width]` (the same channels-first layout from Notebook 01). Photos come in all sorts of sizes, so for each one we:

1. resize so the shorter side is 72 pixels,
2. cut out the 64 × 64 center,
3. turn it into a tensor of values between 0 and 1.

Once they're all the same size, we can stack them into one big tensor. We should get `[800, 3, 64, 64]`.

In [ ]:
image_transform = Compose([Resize(72), CenterCrop(IMAGE_SIZE), ToTensor()])  # same size for every image, so they stack into one tensor
rows = [dataset[first_row[cocoid]] for cocoid in train_ids]
images = torch.stack([image_transform(row["image"].convert("RGB")) for row in rows])
print("image tensor [pair, channel, height, width]:", tuple(images.shape))

Before training anything, let's eyeball a few pairs. The caption above each image is the one that'll sit at the same diagonal spot in the similarity matrix later.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))  # titles line up with images, same order as the tensor
for axis, image, caption in zip(axes, images[:4], captions[:4]):
    axis.imshow(image.permute(1, 2, 0))
    axis.set_title(fill(caption, 28), fontsize=8)
    axis.axis("off")
fig.suptitle("Four real MiniCLIP training pairs")
fig.tight_layout()
plt.show()

**TODO:** Peek at any pair you like. Change `my_pair` (anything from 0 to 799) and re-run. Besides the caption we train on, this also prints every other caption COCO has for that photo.

In [ ]:
# TODO: pick any pair from 0 to 799
my_pair = 4
my_id = train_ids[my_pair]

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(images[my_pair].permute(1, 2, 0))
ax.set_title(f"pair {my_pair}")
ax.axis("off")
plt.tight_layout()
plt.show()

print("caption we train on:", captions[my_pair])
print("all captions for this photo:")
for other_caption in captions_by_id[my_id]:
    print(" -", other_caption)

We only ever train on that first caption. The others describe the same photo just as well, which is worth keeping in mind later when the model "misses".

## 3. How do captions become numbers?

### 3.1 Borrow CLIP's tokenizer

A **tokenizer** splits text into pieces from a fixed vocabulary and gives each piece an integer ID. CLIP uses **BPE** (byte-pair encoding): the vocabulary is built by repeatedly merging character pairs that show up together a lot, so common words end up as one piece and rare words get split into several.

> **Heads up:** Using CLIP's official tokenizer does **not** give our text tower any pretrained knowledge. All we borrow is the vocabulary and the rule for splitting strings. The embedding table we build in Section 4 starts out completely random.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(  # just the splitting rules and vocabulary, no pretrained text encoder
    TOKENIZER_ID, revision=TOKENIZER_REVISION
)
vocabulary_size = len(tokenizer)
sequence_length = tokenizer.model_max_length
print("vocabulary / positions:", vocabulary_size, sequence_length)

Two numbers to note. The first is the vocabulary size: how many different pieces exist. Our text tower will need one learned row for every one of them. The second is 77, the longest sequence CLIP handles.

Now let's tokenize a single caption before doing all 800. Watch for words that turn into several pieces, and punctuation that gets its own piece. The `</w>` glued onto pieces is just CLIP's way of marking "a word ends here".

In [ ]:
first_pieces = tokenizer.tokenize(captions[0])  # BPE pieces are vocabulary entries, nothing learned yet
print("caption:", captions[0])
print("BPE pieces:", first_pieces)

**TODO:** Tokenize your own sentence. Try a made-up word, or a long rare one, and watch it break into pieces.

In [ ]:
# TODO: type any sentence you like
my_sentence = "A skateboarder doing a trick on a ramp."
print("BPE pieces:", tokenizer.tokenize(my_sentence))

### 3.2 Encode and pad every caption

Captions have different lengths, but a tensor has to be a rectangle. So we **pad**: we add filler tokens until every caption is 77 positions long. The tokenizer also adds a start-of-text token at the front and an end-of-text token after the caption.

To tell real tokens from filler, the tokenizer gives us an **attention mask**: `1` for a real position, `0` for padding.

> **Heads up:** CLIP uses the *same* ID for padding and for end-of-text. So you can't tell padding apart just by looking at the IDs. The mask is the only reliable way, and our text tower will lean on it.

In [ ]:
encoded_captions = tokenizer(  # same order as the image tensor
    captions,
    padding="max_length",
    truncation=True,
    max_length=sequence_length,
    return_tensors="pt",
)
tokens = encoded_captions["input_ids"]
token_masks = encoded_captions["attention_mask"]

> **Pause and predict:** We have 800 captions, each padded to 77 positions. What shapes will the token IDs and the mask have?

In [ ]:
print("token IDs [pair, position]:", tuple(tokens.shape))
print("attention mask:", tuple(token_masks.shape))  # IDs and mask describe the same positions

Both are `[800, 77]`. The 800 rows match the image tensor's first dimension, so row `i` is still caption `i`. And the 77 columns are **token positions**, not embedding dimensions. No learned features exist yet, just IDs.

**TODO:** Let's see that padding gotcha for real. Pick a caption with `my_row` and check: are the padding ID and the end-of-text ID the same number?

In [ ]:
# TODO: change which caption to look at (0 to 799)
my_row = 0
print("pad ID / end-of-text ID:", tokenizer.pad_token_id, tokenizer.eos_token_id)
print("real tokens in this caption:", token_masks[my_row].sum().item())
print("first 20 IDs:", tokens[my_row, :20].tolist())

Same number twice. In the ID list, the first ID is the start-of-text token, and (if the caption is short enough to fit in 20) you'll see the end-of-text ID show up right after the caption and then just keep repeating as padding. Only the mask knows which one is the real ending.

### 3.3 Batch them up without breaking pairs

`TensorDataset` ties the image, the token IDs, and the mask together under one index. The `DataLoader` then shuffles *indices*, so an image can never get separated from its caption.

With 800 pairs and batches of 50, that's 800 ÷ 50 = **16** updates per epoch.

In [ ]:
paired_data = TensorDataset(images, tokens, token_masks)  # one index grabs the whole pair
loader_generator = torch.Generator().manual_seed(SEED)  # its own seed, so the shuffle order is reproducible
train_loader = DataLoader(
    paired_data, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator
)
print("batches per epoch:", len(train_loader))

## 4. What do the two towers look like?

A **tower** is just the encoder for one kind of input. CLIP has two: one for images, one for text. Ours are much smaller than CLIP's, but they have the same job: turn their input into one vector.

### 4.1 Image tower: pixels to one vector

Three small convolution blocks, each with stride 2, so every block halves the grid: 64 × 64 → 32 × 32 → 16 × 16 → 8 × 8. Along the way the channels grow from 3 → 32 → 64 → 128, so the later layers can pick up broader visual patterns.

At the end, **global average pooling** averages the whole 8 × 8 grid for each channel. That throws away the spatial layout and leaves one 128-number vector per image.

In [ ]:
class ImageTower(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),  # 64 x 64 -> 32 x 32
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),  # 32 x 32 -> 16 x 16
            nn.Conv2d(64, width, 3, stride=2, padding=1), nn.ReLU(),  # 16 x 16 -> 8 x 8
            nn.AdaptiveAvgPool2d(1),  # average the grid down to one value per channel
        )

    def forward(self, batch_images):
        return self.network(batch_images).flatten(1)  # [B, 128, 1, 1] -> [B, 128]

### 4.2 Text tower: token IDs to one vector

Even simpler. Each token ID looks up its own learned row of 128 numbers (that lookup table is called an **embedding**). Then we average the rows of the real tokens only, using the mask to skip padding. One 128-number vector per caption.

Notice what's missing: word order. Averaging doesn't care what came first. So this is a deliberately simple **bag-of-words** encoder (a continuous bag-of-words, or CBOW-style, average).

> **Something to chew on:** To this tower, *"a dog chasing a cat"* and *"a cat chasing a dog"* are the exact same thing. Hold that thought. You'll test it yourself in TODO 2.

In [ ]:
class TextTower(nn.Module):
    def __init__(self, vocabulary_size, width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, width)  # one row per vocabulary piece, all random at the start

    def forward(self, batch_tokens, batch_mask):
        embedded = self.embedding(batch_tokens)
        weights = batch_mask.unsqueeze(-1).to(embedded.dtype)  # 1 for real tokens, 0 for padding
        token_sum = (embedded * weights).sum(dim=1)  # add up only the real tokens
        token_count = weights.sum(dim=1).clamp_min(1)  # how many real tokens (at least 1, so we never divide by 0)
        return token_sum / token_count  # their average

### 4.3 Meet in a shared space

Both towers now output 128 numbers, but their coordinates have nothing to do with each other. Number 5 of an image vector and number 5 of a text vector don't mean the same thing. So each side gets its own learned **projection** (a linear layer with no bias, like in CLIP) that maps it into a shared 64-dimensional space. Then L2 normalization makes every vector length 1, so a dot product is a cosine similarity, just like in Notebooks 01 and 03.

One more piece: the **temperature**. It's a single number that divides the similarity scores before the softmax. A small temperature makes the scores sharper, so the model gets more confident. CLIP learns it instead of fixing it by hand.

We store it as `log(1 / temperature)` and call `.exp()` whenever we need it. That way it's always positive, no matter where the optimizer pushes it. Starting at 0.07 means the scale starts at 1 / 0.07 ≈ 14.3.

In [ ]:
class MiniCLIP(nn.Module):  # one parent module holding both towers, so one optimizer trains everything
    def __init__(self, vocabulary_size):
        super().__init__()
        self.image_encoder = ImageTower(TOWER_WIDTH)
        self.text_encoder = TextTower(vocabulary_size, TOWER_WIDTH)
        self.image_projection = nn.Linear(TOWER_WIDTH, EMBED_DIM, bias=False)
        self.text_projection = nn.Linear(TOWER_WIDTH, EMBED_DIM, bias=False)
        self.logit_scale = nn.Parameter(torch.tensor(1 / INITIAL_TEMPERATURE).log())  # stored as log(1 / temperature)

    def forward(self, batch_images, batch_tokens, batch_mask):
        image = self.image_projection(self.image_encoder(batch_images))
        text = self.text_projection(self.text_encoder(batch_tokens, batch_mask))
        return F.normalize(image, dim=-1), F.normalize(text, dim=-1)  # unit-length vectors on both sides

Let's build it. The parameter count covers everything we're about to train: both towers, both projections, and the temperature.

In [ ]:
model = MiniCLIP(vocabulary_size).to(device)  # moving the parent moves every submodule
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("trainable parameters:", f"{parameter_count:,}")
print("initial temperature:", model.logit_scale.exp().reciprocal().item())

The temperature should read about 0.07 (float rounding may tack on a few extra digits).

And where do all those parameters live? Almost all of them sit in the text tower's embedding table: one 128-number row for every piece in the vocabulary. The three conv layers and the two projections are tiny next to it.

## 5. What does one training step actually compute?

Before we hand the loss to a training loop, let's walk through it once by hand on our **untrained** model. It's the same calculation as Notebook 03, just with our random towers instead of pretrained CLIP.

Here's Figure 3 from the [CLIP paper](https://arxiv.org/abs/2103.00020) again, since we're about to walk through it one more time. Our towers are tiny, but every line from `I_e` down to `loss` is the same math:

<img src="../assets/diagrams/notebook3/fig3.png" alt="CLIP paper Figure 3: numpy-like pseudocode for the core of CLIP" width="500">

We'll use the first 50 pairs as a fixed **diagnostic batch**: a set we score now, before training, and score again after. Same pairs both times, so the comparison is fair.

In [ ]:
demo_images = images[:BATCH_SIZE].to(device)  # a fixed slice we'll score before and after training
demo_tokens = tokens[:BATCH_SIZE].to(device)
demo_masks = token_masks[:BATCH_SIZE].to(device)
print("images:", tuple(demo_images.shape))
print("token IDs / masks:", tuple(demo_tokens.shape), tuple(demo_masks.shape))

### 5.1 Encode each side separately

Both towers should hand back `[50, 128]`: one row per pair, same width on both sides.

In [ ]:
with torch.inference_mode():  # just measuring a baseline, not training
    image_features = model.image_encoder(demo_images)
    text_features = model.text_encoder(demo_tokens, demo_masks)
print("image / text features:", tuple(image_features.shape), tuple(text_features.shape))

### 5.2 Project and normalize

Projection takes each row from 128 numbers down to 64. Then we normalize, so every row should have length 1. That's what lets us read the dot products in the next step as cosine similarities.

In [ ]:
with torch.inference_mode():  # a snapshot of the untrained model
    image_projected = model.image_projection(image_features)
    text_projected = model.text_projection(text_features)
    image_embeddings = F.normalize(image_projected, dim=-1)
    text_embeddings = F.normalize(text_projected, dim=-1)
print("projected shapes:", tuple(image_embeddings.shape), tuple(text_embeddings.shape))

Shapes look right: `[50, 64]` on both sides. But a shape can't tell you whether the normalization worked. Let's check a few row lengths before we trust any dot product.

In [ ]:
print("first image norms:", image_embeddings.norm(dim=-1)[:3].cpu())
print("first text norms: ", text_embeddings.norm(dim=-1)[:3].cpu())  # values near 1 mean normalization worked

### 5.3 Compare every image with every caption

Multiply `[50, 64]` by the transpose of `[50, 64]` and you get `[50, 50]`. Row `i` is image `i` scored against all 50 captions. Since we kept everything lined up, the true partner of image `i` is caption `i`, which sits right on the **diagonal**.

> **Pause and predict:** Our towers are still random. Should the diagonal already stand out from the rest of the matrix?

In [ ]:
cosine_similarity = image_embeddings @ text_embeddings.T  # row i: image i against all 50 captions
before_scores = cosine_similarity.detach().cpu()
print("similarity matrix:", tuple(cosine_similarity.shape))
print("untrained diagonal mean:", cosine_similarity.diag().mean().item())

It shouldn't. Nothing links image `i` to caption `i` yet, so the diagonal mean is just some number, no more special than the scores off the diagonal. We saved this matrix as `before_scores`, so we can compare against it after training.

### 5.4 Scale the scores and mark the right answers

Now we multiply the scores by the learned scale (1 / temperature). That makes the softmax inside cross-entropy sharper or softer, but it never changes which caption ranks first. A bigger positive number times every score keeps the order the same.

The targets are just `[0, 1, 2, ..., 49]`. That says "row `i` should pick column `i`", which is the diagonal.

In [ ]:
logit_scale = model.logit_scale.exp()  # 1 / temperature, always positive
logits = logit_scale * cosine_similarity
targets = torch.arange(BATCH_SIZE, device=device)
print("logits / targets:", tuple(logits.shape), tuple(targets.shape))

### 5.5 Ask the question both ways

Cross-entropy over the rows treats each image as a 50-way multiple-choice question: which caption is mine? Transpose the matrix and each caption gets its own 50-way question: which image is mine? CLIP counts both directions equally.

> **Pause and predict:** Say the model had no clue at all and gave all 50 options the exact same score. What would each cross-entropy be? *Hint: the right answer gets probability 1/50.*

In [ ]:
image_to_text_loss = F.cross_entropy(logits, targets)  # each image should pick its caption
text_to_image_loss = F.cross_entropy(logits.T, targets)  # each caption should pick its image
print("image→text loss:", image_to_text_loss.item())
print("text→image loss:", text_to_image_loss.item())

With all scores equal, the loss is −ln(1/50) = ln 50 ≈ 3.91. That's the "no idea" level. Expect our untrained numbers to land close to it, often a bit above. Random scores that aren't perfectly flat tend to make things worse, not better, because confidently picking the wrong caption costs more than shrugging.

The two losses answer two different retrieval questions. Their average is the single number we'll actually train on. Let's compute it by hand before tucking it into a function.

In [ ]:
symmetric_loss = (image_to_text_loss + text_to_image_loss) / 2  # neither direction gets special treatment
print("symmetric loss:", symmetric_loss.item())

**TODO:** How much does the temperature really matter? This cell reuses the same untrained cosine scores with a temperature you pick. Try 1.0, 0.07, and 0.01.

In [ ]:
# TODO: try a few temperatures, like 1.0, 0.07, 0.01
my_temperature = 0.07
my_logits = cosine_similarity / my_temperature
my_loss = (F.cross_entropy(my_logits, targets) + F.cross_entropy(my_logits.T, targets)) / 2
print("temperature:", my_temperature, "| symmetric loss:", round(my_loss.item(), 4))
print("same top caption for every image?", torch.equal(my_logits.argmax(dim=1), logits.argmax(dim=1)))

The loss moves a lot, but the top caption for each image stays put. Temperature only changes how *confident* the scores look. With a big temperature like 1.0, the scores look almost flat and the loss sits close to 3.91. With a tiny one, every confident wrong answer gets punished hard. That's exactly why CLIP lets training pick the temperature.

### 5.6 Wrap it in a function

The training loop needs this calculation over and over, so let's give it a name. `clip_loss` does exactly what we just did by hand: cosine scores, positive scale, diagonal targets, two cross-entropies, and their average.

In [ ]:
def clip_loss(image_embeddings, text_embeddings, logit_scale):
    scale = logit_scale.exp()
    logits = scale * image_embeddings @ text_embeddings.T
    targets = torch.arange(logits.shape[0], device=logits.device)  # sized from the batch, in case a batch is smaller (ours never is: 800 / 50 is exactly 16)
    image_loss = F.cross_entropy(logits, targets)
    text_loss = F.cross_entropy(logits.T, targets)
    return (image_loss + text_loss) / 2

Quick sanity check: the function should give the same number as our step-by-step version (up to tiny rounding in the last digits).

In [ ]:
with torch.inference_mode():  # just checking, no gradients needed
    check_loss = clip_loss(image_embeddings, text_embeddings, model.logit_scale)
print("by hand:", symmetric_loss.item(), "| clip_loss:", check_loss.item())

## 6. Time to train

### 6.1 One update

A single update is four moves: clear the old gradients, compute the embeddings and the loss, backpropagate, and nudge the weights. Nothing is frozen, so both towers learn at the same time.

The optimizer is **AdamW**: Adam with weight decay applied as its own separate step. **Weight decay** gently shrinks the weights a little on every update, which helps keep them from growing out of control.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

def training_step(batch_images, batch_tokens, batch_masks):
    optimizer.zero_grad()  # clear last step's gradients
    image, text = model(
        batch_images.to(device), batch_tokens.to(device), batch_masks.to(device)
    )
    loss = clip_loss(image, text, model.logit_scale)
    loss.backward()  # gradients flow into both towers, both projections, and the temperature
    optimizer.step()
    return loss.item()

### 6.2 One pass over all 800 pairs

An **epoch** is one full pass over the data: every mini-batch once. Our function returns the average loss of those 16 batches.

Because the loader shuffles, each image faces a different set of 49 rival captions every epoch. The pairs themselves never change.

In [ ]:
def train_epoch():  # averages the loss over all 16 mini-batches
    model.train()  # training mode (we have no dropout, but it's a good habit)
    losses = []
    for batch_images, batch_tokens, batch_masks in train_loader:
        losses.append(training_step(batch_images, batch_tokens, batch_masks))
    return sum(losses) / len(losses)

### 6.3 Run all 40 epochs

40 epochs × 16 batches = 640 updates. We're keeping the loop plain on purpose: no learning-rate schedule, no early stopping, no data augmentation, no picking the best checkpoint on a validation set. Those can all help, but we don't need any of them to see CLIP's core idea at work.

> **Heads up:** This is the one cell that actually trains, so it's the slowest one in the notebook. Everything after it reuses the trained model, and none of the TODOs ask you to run it again (except the optional TODO 4).

In [ ]:
loss_history = []  # one average loss per epoch
for epoch in range(EPOCHS):
    loss_history.append(train_epoch())

So, did it work? First check: is the loss at the end lower than at the start? If so, the model got better at picking each item's true partner out of its batch.

Peek at the temperature too. It started at 0.07. Did training make the scores sharper (smaller) or softer (bigger)?

In [ ]:
print("first epoch loss:", round(loss_history[0], 4))  # start vs. end of training
print("final epoch loss:", round(loss_history[-1], 4))
print("final temperature:", model.logit_scale.exp().reciprocal().item())

Here's the whole story, one point per epoch. Look at the overall direction, and don't worry if it wiggles a bit. Every epoch shuffles the batches, so the model faces different rivals each time.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4))  # one loss value per epoch
axis.plot(range(1, EPOCHS + 1), loss_history, color="tab:blue")
axis.set(xlabel="epoch", ylabel="symmetric CLIP loss", title="MiniCLIP training loss")
axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 7. Did the model actually learn the pairs?

### 7.1 Score the same 50 pairs again

Same 50 pairs, same scoring rule as Section 5. The only thing that changed is the weights. So whatever difference we see is down to training.

> **Heads up:** These 50 pairs were part of the training data. So this tells us whether the model **fit** them, not whether it works on photos it has never seen.

In [ ]:
model.eval()
with torch.inference_mode():  # no updates, no gradients
    after_image, after_text = model(demo_images, demo_tokens, demo_masks)
    after_scores = (after_image @ after_text.T).cpu()
print("diagonal mean before:", before_scores.diag().mean().item())
print("diagonal mean after: ", after_scores.diag().mean().item())

### 7.2 How often is the right partner ranked first?

**Recall@1** is the fraction of queries where the true partner gets the top score. We check it both ways: along the rows (each image picks a caption, image→text) and down the columns (each caption picks an image, text→image).

In [ ]:
def recall_at_one(scores):  # rows: image->text, columns: text->image
    targets = torch.arange(scores.shape[0])
    image_hits = scores.argmax(dim=1).eq(targets).float().mean()  # best caption for each image
    text_hits = scores.argmax(dim=0).eq(targets).float().mean()  # best image for each caption
    return image_hits.item(), text_hits.item()

Defining the metric doesn't measure anything yet. Now we run the exact same function on the before and after matrices, so both get judged by the same rule.

> **Pause and predict:** With 50 options, random guessing gets the right one 1 time in 50. What Recall@1 do you expect *before* training? And after?

In [ ]:
before_recall = recall_at_one(before_scores)  # same rule for both snapshots
after_recall = recall_at_one(after_scores)
print("R@1 before, image→text / text→image:", before_recall)
print("R@1 after,  image→text / text→image:", after_recall)

Before training, you should see something near chance: 1/50 = 0.02. Since that's out of only 50 items, luck alone can land you on zero, one, or a couple of hits. If the "after" numbers are way higher, the model has learned these pairs. (Training pairs, remember.)

### 7.3 Look at the whole matrix, not just one number

A single number hides a lot, so let's draw the first 16 × 16 corner of both matrices. Both panels share one color range. That way, if the right one has a brighter diagonal, it's a real difference and not just two plots auto-scaling their colors differently.

In [ ]:
before_view = before_scores[:16, :16].numpy()
after_view = after_scores[:16, :16].numpy()
color_min = min(before_view.min(), after_view.min())  # shared limits so the two panels are comparable
color_max = max(before_view.max(), after_view.max())

Now draw both panels side by side. If training worked, the left panel should look like noise and the right one should have a bright diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")  # both panels use the shared limits from above
for axis, matrix, title in zip(axes, [before_view, after_view], ["Before training", "After training"]):
    view = axis.imshow(matrix, cmap="magma", vmin=color_min, vmax=color_max)
    axis.set(xlabel="caption position", ylabel="image position", title=title)
fig.colorbar(view, ax=axes, label="cosine similarity")
plt.show()

### 7.4 Read some actual retrievals

Numbers can't show you what a mistake *looks* like. So for the first five images, let's print the caption the model ranks first, and mark whether it's the image's true partner.

> **Heads up:** "different" doesn't automatically mean wrong. COCO captions are pretty generic, and another photo's caption can describe this one perfectly well. Read the caption and judge for yourself.

In [ ]:
predicted_captions = after_scores.argmax(dim=1)  # highest score = rank one
for image_index in range(5):
    caption_index = predicted_captions[image_index].item()
    status = "paired" if caption_index == image_index else "different"
    print(f"image {image_index} → caption {caption_index} ({status})")
    print(" ", captions[caption_index])

## Recap

That's the whole trip. We built a tiny CLIP and trained it from scratch:

```text
64 × 64 image ─→ ImageTower (3 convs + avg pool) ─→ [128] ─→ projection ─→ unit vector [64] ──────┐
                                                                                                  ├─→ [50, 50] scores × 1/temperature
caption ─→ 77 token IDs ─→ TextTower (masked average) ─→ [128] ─→ projection ─→ unit vector [64] ─┘
                                                                                                  ↓
                                                           symmetric cross-entropy, diagonal = targets
```

**The big ideas:**

- A CLIP-style model learns from **pair positions**. The diagonal supplies the target for every row and every column.
- Normalized image and text vectors let one similarity matrix serve both retrieval directions.
- The symmetric loss trains both directions at once, and its gradient updates both towers, both projections, and the temperature.
- Falling training loss and higher Recall@1 on training pairs show that the model **fit** these examples. They don't yet show it works on new images.

### Check your understanding

1. Why did we sample *photo IDs* instead of picking 800 random rows from the dataset?
2. Suppose we doubled the logit scale (halved the temperature) without retraining. Would the loss change? Would Recall@1?
3. Our after-training Recall@1 is measured on the first 50 training pairs. Why isn't a high number there proof of a good model?

<details>
<summary>Show answers</summary>

1. COCO has several caption rows per photo. Random rows could put the same photo into a batch twice, and then one of its "wrong" captions would actually be correct. The loss would push apart two things that truly match.
2. The loss would change, since the softmax gets sharper. Recall@1 wouldn't: multiplying every score by the same positive number never changes which one is biggest.
3. The model has seen those exact pairs over and over during training, so it may just have memorized them. To test generalization you'd score pairs it never trained on. Also, it's only picking out of 50 options, which is a lot easier than searching a big collection.

</details>

## Try it yourself

TODOs 1 to 3 reuse the trained model, so they're quick. Just make sure you've run the notebook through Section 7 first. TODO 4 is optional and does retrain.

### TODO 1: Search with your own caption

Write a caption, run it through the trained text tower, and see which of the 50 diagnostic images score highest. The cell runs as-is, but the fun part is changing `my_caption`.

In [ ]:
# TODO: write your own caption and re-run
my_caption = "a plate of food on a table"
my_encoded = tokenizer(my_caption, padding="max_length", truncation=True,
                       max_length=sequence_length, return_tensors="pt")

with torch.inference_mode():
    my_text = model.text_projection(model.text_encoder(
        my_encoded["input_ids"].to(device), my_encoded["attention_mask"].to(device)))
    my_text = F.normalize(my_text, dim=-1)
    my_scores = (after_image @ my_text.T).squeeze(1).cpu()  # one score per diagnostic image
my_top = my_scores.topk(3).indices.tolist()

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for axis, image_index in zip(axes, my_top):
    axis.imshow(images[image_index].permute(1, 2, 0))
    axis.set_title(fill(captions[image_index], 28), fontsize=8)
    axis.axis("off")
fig.suptitle(f"Top 3 of 50 for: {my_caption}")
fig.tight_layout()
plt.show()

Each title is that image's own training caption, so you can see why the model might have picked it.

<details>
<summary>Hint</summary>

Only words that actually showed up in the 800 training captions got useful embeddings. A word the model never saw keeps (almost exactly) its random starting row, so it just adds noise to the average. If your results look random, try building a caption out of words from the training pairs, like the ones you saw in Section 2. And with only 800 pairs, don't expect miracles.

</details>

### TODO 2: Can the text tower tell who's chasing whom?

Back in Section 4.2 we said the text tower ignores word order. Let's check. Two sentences, same words, different order:

In [ ]:
# TODO: try your own pair of sentences with the same words in a different order
sentence_a = "a dog chasing a cat"
sentence_b = "a cat chasing a dog"
pair_encoded = tokenizer([sentence_a, sentence_b], padding="max_length", truncation=True,
                         max_length=sequence_length, return_tensors="pt")

with torch.inference_mode():
    pair_vectors = F.normalize(model.text_projection(model.text_encoder(
        pair_encoded["input_ids"].to(device), pair_encoded["attention_mask"].to(device))), dim=-1)
print("cosine similarity:", (pair_vectors[0] @ pair_vectors[1]).item())

You should get 1.0, give or take tiny rounding: the same set of tokens averages to the same vector. Now change a single word in one sentence and see how far the similarity drops.

> **Something to chew on:** Real CLIP's text encoder is a transformer with position embeddings, so in principle it *can* tell these two apart. How well it actually does is a whole research topic of its own.

### TODO 3: Give the model a few more guesses

Recall@1 is strict: only the very top pick counts. **Recall@k** counts a hit if the true partner lands anywhere in the top `k`. Random guessing gets k/50, so 0.1 for `k = 5`.

In [ ]:
# TODO: change k (1 to 50) and re-run
my_k = 5

def recall_at_k(scores, k):
    targets = torch.arange(scores.shape[0])
    image_hits = (scores.topk(k, dim=1).indices == targets[:, None]).any(dim=1).float().mean()  # true caption in each row's top k?
    text_hits = (scores.topk(k, dim=0).indices == targets[None, :]).any(dim=0).float().mean()  # true image in each column's top k?
    return image_hits.item(), text_hits.item()

print(f"R@{my_k} before, image→text / text→image:", recall_at_k(before_scores, my_k))
print(f"R@{my_k} after,  image→text / text→image:", recall_at_k(after_scores, my_k))

Sanity check: with `my_k = 1` you should get the same numbers as `recall_at_one`. And with `my_k = 50` everything hits, since every item is in the top 50 of 50.

### TODO 4 (optional, slow): Change a hyperparameter and retrain

This one **does** retrain the whole model, so only do it when you've got a few minutes. Pick **one** value in the settings cell in Section 1.3, change it, then restart the kernel and run all cells. Write down your old numbers first (final loss, final temperature, Recall@1) so you have something to compare against.

A few ideas:

- `INITIAL_TEMPERATURE = 1.0`: does training pull the temperature back down, or does the model struggle?
- `BATCH_SIZE = 20` or `100`: fewer or more rivals per image. Heads up, this also changes the size of the diagnostic batch, so compare Recall@1 to the new chance level, 1 / `BATCH_SIZE`.
- `EPOCHS = 10` or `80`: does the loss curve flatten out, or is it still dropping at the end?
- `EMBED_DIM = 16`: does a much smaller shared space hurt Recall@1?

<details>
<summary>Going further: test on photos the model never saw</summary>

This part doesn't need retraining.
Want to actually test generalization? Grab a few more photo IDs with `[all_ids[index] for index in order[TRAIN_SIZE:TRAIN_SIZE + 50].tolist()]` (photos the model never trained on), build their image and token tensors the same way as in Sections 2 and 3, and run `recall_at_one` on their after-training similarity matrix. Compare that to the training-pair numbers.

</details>

---

**Up next, Notebook 05:** no training at all. We'll take a frozen, pretrained CLIP and turn a handful of written class descriptions into a classifier for CIFAR-10 test images it has never seen. That's zero-shot classification.